In [ ]:
from google.colab import files
uploaded = files.upload()


<IPython.core.display.HTML object>

Saving spa.txt to spa.txt


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_data(path, max_samples=10000):
    pairs = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                pairs.append((parts[0].lower(), parts[1].lower()))
    return pairs[:max_samples]

pairs = load_data("spa.txt")
print("Total pairs:", len(pairs))


Total pairs: 10000


In [ ]:
def build_vocab(sentences):
    counter = Counter()
    for sent in sentences:
        counter.update(sent.split())

    vocab = {"<pad>":0, "<sos>":1, "<eos>":2, "<unk>":3}
    for word in counter:
        vocab[word] = len(vocab)
    return vocab

src_sentences = [p[0] for p in pairs]
trg_sentences = [p[1] for p in pairs]

src_vocab = build_vocab(src_sentences)
trg_vocab = build_vocab(trg_sentences)


In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, pairs, src_vocab, trg_vocab, max_len=20):
        self.pairs = pairs
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab
        self.max_len = max_len

    def encode(self, sentence, vocab):
        tokens = ["<sos>"] + sentence.split() + ["<eos>"]
        ids = [vocab.get(tok, vocab["<unk>"]) for tok in tokens]
        return torch.tensor(ids[:self.max_len])

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src, trg = self.pairs[idx]
        return self.encode(src, self.src_vocab), self.encode(trg, self.trg_vocab)


In [ ]:
train_pairs, test_pairs = train_test_split(pairs, test_size=0.2, random_state=42)
val_pairs, test_pairs = train_test_split(test_pairs, test_size=0.5)

def collate_fn(batch):
    src_batch, trg_batch = zip(*batch)
    src_batch = nn.utils.rnn.pad_sequence(src_batch, padding_value=0)
    trg_batch = nn.utils.rnn.pad_sequence(trg_batch, padding_value=0)
    return src_batch.to(device), trg_batch.to(device)

train_loader = DataLoader(
    TranslationDataset(train_pairs, src_vocab, trg_vocab),
    batch_size=32, shuffle=True, collate_fn=collate_fn
)


In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hid_dim)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.lstm(embedded)
        return hidden, cell


In [ ]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hid_dim)
        self.fc = nn.Linear(hid_dim, output_dim)

    def forward(self, input, hidden, cell):
        input = input.unsqueeze(0)
        embedded = self.embedding(input)
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction = self.fc(output.squeeze(0))
        return prediction, hidden, cell


In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        trg_len, batch_size = trg.shape
        trg_vocab_size = self.decoder.fc.out_features

        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(device)
        hidden, cell = self.encoder(src)

        input = trg[0]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[t] = output
            top1 = output.argmax(1)
            input = trg[t] if random.random() < teacher_forcing_ratio else top1

        return outputs


In [ ]:
encoder = Encoder(len(src_vocab), 256, 512).to(device)
decoder = Decoder(len(trg_vocab), 256, 512).to(device)

model = Seq2Seq(encoder, decoder).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=0)


In [ ]:
for epoch in range(10):
    model.train()
    epoch_loss = 0

    for src, trg in train_loader:
        optimizer.zero_grad()
        output = model(src, trg)

        output = output[1:].reshape(-1, output.shape[-1])
        trg = trg[1:].reshape(-1)

        loss = criterion(output, trg)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {epoch_loss/len(train_loader):.4f}")


Epoch 1, Loss: 5.1969
Epoch 2, Loss: 3.7788
Epoch 3, Loss: 2.8382
Epoch 4, Loss: 2.0814
Epoch 5, Loss: 1.4633
Epoch 6, Loss: 1.0399
Epoch 7, Loss: 0.7861
Epoch 8, Loss: 0.6337
Epoch 9, Loss: 0.5564
Epoch 10, Loss: 0.5057
